# Attribute extraction: size, storage and pack quantity

Fills `data/attributes_submission_template.csv` for the 200 items in `data/attributes_sample.csv`. These are the only two data files this notebook reads.

**Method.** Regex rules written from the brief and the 200 descriptions, applied one row at a time with no fitted state. The descriptions are short and use a limited set of patterns, so rules are free to run, repeatable and easy to audit. A value is filled in only when the description states it.

**Result.** `size` is filled on 167 rows, `storage` on 47 and `pack_qty` on 114. Every other cell is blank because the description doesn't state a value.

**Estimate for 10,000 new items.** The extractor itself takes under a second, with no API calls. The real cost is people checking the output: about 5 to 9 hours. Section 5 shows the working, and its assumptions can be changed.

## 1. Setup

Only the ids and header are used from the template. Any values already in it are replaced in section 4.

In [2]:
import re
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 250)

DATA = Path("..") / "data"  # notebook lives in notebooks/
TEMPLATE_PATH = DATA / "attributes_submission_template.csv"

sample = pd.read_csv(DATA / "attributes_sample.csv", dtype=str)
template = pd.read_csv(TEMPLATE_PATH, dtype=str)

assert sample["id"].tolist() == template["id"].tolist(), "ids must match, in the same order"
print(sample.shape, template.columns.tolist())

(200, 3) ['id', 'size', 'storage', 'pack_qty']


## 2. Patterns and `extract()`

Each description goes through the same steps:

1. **Clean.** Lowercase, drop supplier housekeeping tokens such as `*** USE ITEM 46932 ***` and `REF#A10116`, and split flags glued to a word (`PK6PROMO`).
2. **Storage.** Read from the whole cleaned text, using the terms in the brief plus the abbreviations in the sample (`fzn`, `chill`, `CHLD`, `AMB`).
3. **Remove what isn't a size.** Percentages, wattages, thread counts, gsm and garment sizes.
4. **Size and pack.** Each match is blanked out so a number can't be read twice. Order matters: `46X33CM` has to be read as a two-dimension size before `500X12GM` can be read as a pack of 500 at 12g.
5. **Review flag.** Any number left over is flagged for a person to check. The flag isn't written to the CSV.

The unit spellings and housekeeping patterns are adapted from `data_preprocessing.ipynb`. Sizes keep the unit stated in the description, written as in the brief: `1.5L`, `500g`, `180x180cm`.

In [3]:
# Every spelling maps to how the unit is written in the output. Any metric unit is allowed.
# A bare "m" is left out on purpose: it collides with garment size M, which isn't a size.
UNITS = {
    "kg": "kg", "kgs": "kg", "kilo": "kg", "kilos": "kg",
    "g": "g", "gm": "g", "gms": "g", "gr": "g", "grm": "g", "gram": "g", "grams": "g",
    "mg": "mg",
    "l": "L", "lt": "L", "ltr": "L", "ltrs": "L",
    "litre": "L", "litres": "L", "liter": "L", "liters": "L",
    "ml": "ml", "mls": "ml", "cl": "cl",
    "mm": "mm", "cm": "cm",
    "mtr": "m", "mtrs": "m", "metre": "m", "metres": "m", "meter": "m", "meters": "m",
}
LENGTH_UNITS = {"mm", "cm", "m"}


def _any_of(words) -> str:
    """Regex alternation, longest first so 'ltr' is tried before 'l'."""
    return "|".join(sorted(words, key=len, reverse=True))


NUM = r"\d+(?:\.\d+)?"
START = r"(?<![\d.])"  # a number starts here, so '1.5' is never read as '5'
NOT_YEAR = r"(?!(?:19|20)\d\d(?!\d))"
PACK_WORDS = "ctn|cs|case|carton|box|pk|pack"
EACH_WORDS = r"pcs?|pieces?|ea|each|btls?|bottles?"
FLAGS = "new|promo|disc|spec|imp"

ANY_UNIT = _any_of(UNITS)
CONTENT_UNIT = _any_of(k for k, v in UNITS.items() if v not in LENGTH_UNITS)
LENGTH_UNIT = _any_of(k for k, v in UNITS.items() if v in LENGTH_UNITS)

# Words that can be glued straight after a unit: 500GCTN/6, 330MLCTN24, 250gx10
GLUED_WORD = (
    rf"(?:{PACK_WORDS}|pcs?|ea|btl|x\d*|amb|ambient|frz|fzn|frozen|chilled|chill|chld"
    r"|refrigerated|dry|shelf|iqf)(?![a-z])"
)


def _unit(units: str) -> str:
    """A unit that ends at a non-letter or a glued word. '10 LAMB' is never 10 L + AMB."""
    return rf"(?!lamb(?![a-z]))(?P<unit>{units})(?:(?![a-z])|(?={GLUED_WORD}))"


HOUSEKEEPING = [
    r"\*+\s*(?:use|sub)[^*]*\*+",  # *** USE ITEM 46932 ***, *SUB 90720*
    r"ref\s*#\s*[a-z]?\d*",  # REF#A10116
    r"\b(?:item|plu)\s*\d+",  # ITEM 28533, PLU 9806
    r"\bline\s*\d{4,}",  # LINE 83171, but not the brand in "IVORY LINE 90 ML"
    r"\*+",  # stray asterisks
]

# The brief says these are not sizes
NOT_A_SIZE = [
    rf"{START}{NUM}\s*-?\s*%",  # 3.5% FAT, 13% ABV
    rf"{START}{NUM}\s*-?\s*(?:gsm|tc|t/c|thread\s*count|watts?|kw|w)(?![a-z])",  # 500GSM, 300TC, 2000W
    r"(?<![a-z0-9])\d?x{1,3}[sl](?![a-z0-9])",  # garment sizes: XL, 2XL, XS
]

# Word boundaries keep out false friends: chilli is not chill, and lamb and bamboo are not amb.
# "freeze dried", "freezer", "cold pressed" and "fresh" match nothing here, so they stay blank.
STORAGE = {
    "frozen": re.compile(r"frozen|frz|fzn|iqf"),
    "chilled": re.compile(r"chilled|chld|refrigerated|(?<![a-z])chill(?![a-z])"),
    "ambient": re.compile(r"ambient|(?<![a-z])amb(?![a-z])|shelf\W*stable|dry\W*store"),
}

# Numbers that are fine to leave unplaced: ages and vintages (Rum 12 year, Shiraz 2022)
FINE_TO_LEAVE = [
    rf"{START}\d{{1,2}}\s*-?\s*(?:years?|yrs?|yo)(?![a-z])",
    rf"{START}(?:19|20)\d\d(?![\d.])",
]

THREE_D = re.compile(rf"{START}{NUM}\s*x\s*{NUM}\s*x\s*{NUM}\s*(?:{LENGTH_UNIT})(?![a-z])")
DOZEN = re.compile(rf"{START}(?P<n>\d+)\s*[-/]?\s*(?:dozen|doz)(?![a-z])")  # 1 doz, 1dozen, 2/DOZEN
TWO_D = re.compile(  # 180X180CM, 30 X 30 CM, 330x330mm
    rf"{START}(?P<a>{NUM})\s*(?:{LENGTH_UNIT})?[\s/-]*x[\s/-]*(?P<b>{NUM})\s*[-/]?\s*" + _unit(LENGTH_UNIT)
)
PACK_THEN_SIZE = re.compile(  # 6x1kg, 12/750ml, 24-x 60gm, 12 x/100gm, 500X12GM
    rf"{START}{NOT_YEAR}(?P<n>\d{{1,4}})[\s/-]*(?:x|/|of)[\s/-]*(?P<q>{NUM})\s*[-/]?\s*" + _unit(CONTENT_UNIT)
)
SIZE_THEN_PACK = re.compile(  # 250g x10, 500ML X12, 1.5 LTR X6
    rf"{START}(?P<q>{NUM})\s*[-/]?\s*" + _unit(CONTENT_UNIT)
    + rf"[\s/-]*x\s*(?P<n>\d+)(?![\d.]*\s*(?:{ANY_UNIT})(?![a-z]))"
)
SINGLE_SIZE = re.compile(rf"{START}(?P<q>{NUM})\s*[-/]?\s*" + _unit(ANY_UNIT))  # 1KG, 20 litre, 19 cm

# Pack forms with no size attached, most specific first
PACK_ONLY = [
    # 12 EA/CS, 6/cs, 1000/CTN, 100pcs/ctn, 1000 per case
    re.compile(
        rf"{START}(?P<n>\d+)\s*-?\s*(?:{EACH_WORDS})?\s*(?:/\s*per[\s/-]*|per[\s/-]*|/\s*)(?:{PACK_WORDS})(?![a-z])"
    ),
    # ctn 6, CTN/6, PK/6, BOX 24, case of-100; ctn and pk can be glued to a word (glovsPK1000)
    re.compile(
        r"(?:(?<![a-z])(?:cs|case|carton|box|pack)|ctn|pk)\s*[/-]*\s*(?:of)?\s*[/-]*\s*(?P<n>\d+)(?![\d.])"
    ),
    re.compile(rf"{START}(?P<n>\d+)\s*[-/]?\s*(?:{EACH_WORDS}|pk|packs?)(?![a-z])"),  # 100 pcs, 6 pack
    re.compile(rf"\(\s*(?P<n>\d{{1,4}})\s*(?:{EACH_WORDS}|pk)?\s*\)"),  # (4), (6 btl)
    re.compile(rf"{START}x\s*(?P<n>\d+)(?![\d.])"),  # wine x6, 330x330mm x1000
    re.compile(rf"{START}(?P<n>\d+)\s*x(?![a-z0-9])"),  # 750ml 6x
]

In [4]:
def clean(description: str) -> str:
    """Lowercase, drop housekeeping tokens and split flags glued to a word."""
    text = str(description).lower().replace("×", "x")
    for pattern in HOUSEKEEPING:
        text = re.sub(pattern, " ", text)
    return re.sub(rf"(?<=[a-z0-9])({FLAGS})(?![a-z])", r" \1", text)  # PK6PROMO, 1KGNEW


def _take(pattern: re.Pattern, text: str) -> tuple[re.Match | None, str]:
    """Return the first match and the text with that span blanked out."""
    match = pattern.search(text)
    if match is None:
        return None, text
    return match, text[: match.start()] + " " + text[match.end():]


def _number(text: str) -> str:
    return text.rstrip("0").rstrip(".") if "." in text else text


def extract(description: str) -> dict:
    """Extract size, storage and pack_qty from one description. Blank means not stated."""
    text = clean(description)
    review = []

    stored = [name for name, pattern in STORAGE.items() if pattern.search(text)]
    if len(stored) > 1:
        review.append("storage conflict")

    for pattern in NOT_A_SIZE:
        text = re.sub(pattern, " ", text)

    contents, lengths, packs = [], [], []

    match, text = _take(THREE_D, text)  # the brief covers two dimensions at most
    if match:
        review.append("three dimensions")

    match, text = _take(DOZEN, text)
    if match:
        packs.append(int(match["n"]) * 12)

    match, text = _take(TWO_D, text)
    if match:
        lengths.append(f"{_number(match['a'])}x{_number(match['b'])}{UNITS[match['unit']]}")

    match, text = _take(PACK_THEN_SIZE, text)
    if match is None:
        match, text = _take(SIZE_THEN_PACK, text)
    if match:
        packs.append(int(match["n"]))
        contents.append(_number(match["q"]) + UNITS[match["unit"]])

    while True:
        match, text = _take(SINGLE_SIZE, text)
        if match is None:
            break
        unit = UNITS[match["unit"]]
        (lengths if unit in LENGTH_UNITS else contents).append(_number(match["q"]) + unit)

    for pattern in PACK_ONLY:
        while True:
            match, text = _take(pattern, text)
            if match is None:
                break
            packs.append(int(match["n"]))

    contents, lengths = list(dict.fromkeys(contents)), list(dict.fromkeys(lengths))
    packs = list(dict.fromkeys(packs))
    if len(contents) > 1 or len(lengths) > 1 or (contents and lengths):
        review.append("size conflict")
    if len(packs) > 1:
        review.append("pack conflict")

    for pattern in FINE_TO_LEAVE:
        text = re.sub(pattern, " ", text)
    if re.search(r"\d", text):
        review.append("unplaced number")

    return {
        "size": (contents or lengths or [""])[0],  # a weight or volume wins over a length
        "storage": stored[0] if stored else "",
        "pack_qty": str(packs[0]) if packs else "",
        "review": ", ".join(review),
    }

## 3. Hand check

The expected values below were read from each description by hand, not copied from the parser's output. The assertions fail loudly if a pattern change breaks one.

The first list covers the brief's own examples, what it says not to infer, what it says isn't a size, and words that look like storage terms but aren't.

In [5]:
BRIEF_CASES = [
    # description, size, storage, pack_qty
    # size and pack examples from the brief
    ("Water 330ml", "330ml", "", ""),
    ("Wine 1.5L", "1.5L", "", ""),
    ("Butter 500g", "500g", "", ""),
    ("Fish 2.5kg", "2.5kg", "", ""),
    ("Plate 270mm", "270mm", "", ""),
    ("Towel 70x140cm", "70x140cm", "", ""),
    ("Flour 6x1kg", "1kg", "", "6"),
    ("Wine 12/750ml", "750ml", "", "12"),
    ("Soap ctn 6", "", "", "6"),
    ("Fork 1 doz", "", "", "12"),
    # storage terms from the brief
    ("Peas frozen", "", "frozen", ""),
    ("Peas FRZ", "", "frozen", ""),
    ("Prawn IQF", "", "frozen", ""),
    ("Milk chilled", "", "chilled", ""),
    ("Ham refrigerated", "", "chilled", ""),
    ("Rice ambient", "", "ambient", ""),
    ("Beans shelf stable", "", "ambient", ""),
    ("Rice dry store", "", "ambient", ""),
    # don't infer
    ("Ice cream 5L", "5L", "", ""),
    ("Cleaner", "", "", ""),
    # not sizes
    ("Milk 3.5% fat", "", "", ""),
    ("Kettle 2000W 1.7L", "1.7L", "", ""),
    ("Sheet 300TC 183X203CM", "183x203cm", "", ""),
    ("Sheet 400 thread count", "", "", ""),
    ("Towel 500GSM 70X140CM", "70x140cm", "", ""),
    ("Chef jacket XL", "", "", ""),
    ("Gloves L PK 100", "", "", "100"),
    ("Gloves 2XL PK 100", "", "", "100"),
    # false friends
    ("green chilli 1 kg", "1kg", "", ""),
    ("LAMB LEG 2KG", "2kg", "", ""),
    ("10 LAMB", "", "", ""),
    ("Bamboo skewer PK 100", "", "", "100"),
    ("Freezer bag PK 50", "", "", "50"),
    ("refrigerator thermometer", "", "", ""),
    ("Coffee freeze dried 1kg", "1kg", "", ""),
    ("Juice cold pressed 1L", "1L", "", ""),
    ("Lemongrass fresh 1kg", "1kg", "", ""),
]

failures = []
for description, size, storage, pack_qty in BRIEF_CASES:
    got = extract(description)
    if (got["size"], got["storage"], got["pack_qty"]) != (size, storage, pack_qty):
        failures.append((description, size, storage, pack_qty, got))
for failure in failures:
    print("expected", failure[1:4], "got", failure[4], "for", failure[0])
assert not failures, "some examples parse differently from the hand-checked reading"
print(f"all {len(BRIEF_CASES)} brief examples and traps pass")

all 37 brief examples and traps pass


The second list holds the hardest real rows in the sample: glued text, unusual pack forms, and numbers that are neither a size nor a pack.

In [6]:
HARD_ROWS = {
    # id: size, storage, pack_qty
    "AT0003": ("100g", "ambient", "6"),
    "AT0007": ("750ml", "ambient", "6"),
    "AT0016": ("330ml", "", "24"),
    "AT0027": ("500g", "", "6"),
    "AT0038": ("25kg", "ambient", ""),
    "AT0042": ("1kg", "", "24"),
    "AT0052": ("", "chilled", ""),
    "AT0065": ("", "", "100"),
    "AT0071": ("30x30cm", "", "10"),
    "AT0074": ("1L", "chilled", ""),
    "AT0086": ("33x33cm", "", "100"),
    "AT0100": ("330x330mm", "", "1000"),
    "AT0103": ("60g", "", "24"),
    "AT0133": ("250ml", "", "12"),
    "AT0137": ("1L", "", "12"),
    "AT0138": ("25kg", "", "10"),
    "AT0142": ("750ml", "ambient", "12"),
    "AT0143": ("1kg", "chilled", ""),
    "AT0146": ("40x70cm", "", ""),
    "AT0150": ("100g", "", "12"),
    "AT0156": ("1L", "", ""),
    "AT0164": ("200g", "", "6"),
    "AT0180": ("1kg", "frozen", "6"),
    "AT0182": ("500g", "ambient", "25"),
    "AT0189": ("750ml", "", "6"),
    "AT0191": ("12g", "", "500"),
    "AT0195": ("500g", "chilled", "40"),
    "AT0197": ("", "", "1000"),
    "AT0199": ("2kg", "chilled", "6"),
}

descriptions = sample.set_index("id")["description"]
hard = pd.DataFrame(
    [(i, descriptions[i], *values) for i, values in HARD_ROWS.items()],
    columns=["id", "description", "size", "storage", "pack_qty"],
)
parsed = pd.DataFrame([extract(d) for d in hard["description"]])
mismatched = (hard[["size", "storage", "pack_qty"]] != parsed[["size", "storage", "pack_qty"]]).any(axis=1)
assert not mismatched.any(), hard[mismatched].join(parsed, rsuffix="_got").to_string()
print(f"all {len(hard)} hard rows match")
hard

all 29 hard rows match


,id,description,size,storage,pack_qty
0,AT0003,HOT DOG ROLL-MILK AMB-100G-CTN/6,100g,ambient,6
1,AT0007,Metro-ValuePinot-Grigio house-pour ambient 750ml 6x,750ml,ambient,6
2,AT0016,Chef's Choice Tonic Water diet 24/330ml,330ml,,24
3,AT0027,PRWN 500GCTN/6,500g,,6
4,AT0038,PANTRY-PRO-RICE/DRY/STORE 25 KG,25kg,ambient,
5,AT0042,*** USE-ITEM 33777 *** YSTERS/PRIME/KITCHEN 1KG/HALF SHELL/2/DOZEN,1kg,,24
6,AT0052,pizza base Pantry Pro CHLD,,chilled,
7,AT0065,serenia shower cap individually wrapped case of-100,,,100
8,AT0071,"DISCPOOL TOWEL, 30 X 30 CM, 10 EA/CS",30x30cm,,10
9,AT0074,ROYAL TABLE FULL CRMMILK 3.5% FAT CHILL 1 LITRE,1L,chilled,


## 4. Run on the 200 rows

The output is validated before it's written, and blank cells stay blank in the CSV.

In [7]:
extracted = pd.DataFrame([extract(d) for d in sample["description"]])
submission = pd.concat([sample[["id"]], extracted[["size", "storage", "pack_qty"]]], axis=1)

SIZE_FORMAT = rf"{NUM}(?:x{NUM})?(?:{_any_of(set(UNITS.values()))})"
filled_sizes = submission.loc[submission["size"] != "", "size"]

assert submission.columns.tolist() == template.columns.tolist()
assert submission["id"].tolist() == template["id"].tolist()
assert submission["storage"].isin(["", "frozen", "chilled", "ambient"]).all()
assert submission["pack_qty"].str.fullmatch(r"\d*").all(), "pack_qty must be whole numbers"
assert filled_sizes.str.fullmatch(SIZE_FORMAT).all(), "sizes must be a number and a unit"

submission.to_csv(TEMPLATE_PATH, index=False)

print("rows written:", len(submission))
print("filled:", (submission[["size", "storage", "pack_qty"]] != "").sum().to_dict())
print("storage:", submission.loc[submission["storage"] != "", "storage"].value_counts().to_dict())
print("flagged for review:", (extracted["review"] != "").sum())

rows written: 200
filled: {'size': 167, 'storage': 47, 'pack_qty': 114}
storage: {'ambient': 22, 'chilled': 13, 'frozen': 12}
flagged for review: 0


No rows are flagged, as expected, because the rules were written from these rows.

Every row in the table below was also read against its description by hand, with no disagreements.

In [8]:
sample[["id", "description"]].join(submission[["size", "storage", "pack_qty"]]).head()

,id,description,size,storage,pack_qty
0,AT0001,"IVORY LINE, CUP,12 EA/CS",,,12
1,AT0002,HARBOUR SELECT SOY SAUCCE SHELF STABLE 1LT 6/CS,1L,ambient,6
2,AT0003,HOT DOG ROLL-MILK AMB-100G-CTN/6,100g,ambient,6
3,AT0004,Juice dry store 2LT PLU 22208,2L,ambient,
4,AT0005,"RUNNER, 180X180CM",180x180cm,,


## 5. Cost and time for 10,000 items

Machine time is measured by running `extract()` on the 200 rows repeated to 10,000. The people-time inputs are assumptions, set as variables so they can be changed. The 200 rows can't give a review rate, because rules written from them flag none, so 5% is a planning figure.

In [8]:
ITEMS = 10_000

# Assumptions: change these
REVIEW_SHARE = 0.05  # share of new rows flagged for review
REVIEW_SECONDS = 30  # per flagged row
SPOT_CHECK_ROWS = 200  # random rows checked for errors the flags don't catch
SPOT_CHECK_SECONDS = 15  # per row
NEW_FORMAT_HOURS = 4  # allowance for adding rules, if new formats appear

# Measured: median of 5 runs
rows = sample["description"].tolist() * (ITEMS // len(sample))
timings = []
for _ in range(5):
    start = time.perf_counter()
    for description in rows:
        extract(description)
    timings.append(time.perf_counter() - start)
machine_seconds = sorted(timings)[len(timings) // 2]

review_hours = ITEMS * REVIEW_SHARE * REVIEW_SECONDS / 3600
spot_check_hours = SPOT_CHECK_ROWS * SPOT_CHECK_SECONDS / 3600
low_hours = review_hours + spot_check_hours
high_hours = low_hours + NEW_FORMAT_HOURS


pd.DataFrame(
    [
        ("Run the extractor", f"Measured on {len(rows):,} rows, no API calls", f"{machine_seconds:.2f} s"),
        (
            "Review flagged rows",
            f"{REVIEW_SHARE:.0%} of rows ({ITEMS * REVIEW_SHARE:,.0f}) at {REVIEW_SECONDS} s each",
            f"{review_hours:.1f} h",
        ),
        (
            "Spot-check for errors",
            f"{SPOT_CHECK_ROWS} random rows at {SPOT_CHECK_SECONDS} s each",
            f"{spot_check_hours:.1f} h",
        ),
        (
            "Add rules for new formats",
            "Allowance, if new formats appear",
            f"up to {NEW_FORMAT_HOURS} h",
        ),
        (
            "Total",
            "People's time",
            f"{low_hours:.0f} to {high_hours:.0f} h",
        ),
    ],
    columns=["item", "basis", "time"],
)


,item,basis,time
0,Run the extractor,"Measured on 10,000 rows, no API calls",0.30 s
1,Review flagged rows,5% of rows (500) at 30 s each,4.2 h
2,Spot-check for errors,200 random rows at 15 s each,0.8 h
3,Add rules for new formats,"Allowance, if new formats appear",up to 4 h
4,Total,People's time,5 to 9 h


**About 5 to 9 hours of people's time**, which is one to two working days. The machine cost is effectively nothing, so the estimate depends almost entirely on the review rate. The timing covers `extract()` only, and reading and writing the CSV adds very little.

## 6. Limitations

- **Accuracy on new items isn't measured.** The rules were written from these 200 rows, so matching them says little about new descriptions. The spot-check in section 5 is how accuracy on new items would actually be measured.
- **Formats not in the sample go blank.** A new layout or abbreviation may be missed. If a number is left over, the row is flagged for review.
- **Some misses aren't flagged.** A storage word spelled in a way the rules don't know leaves `storage` blank with no flag, because no number is left over. Only the spot-check would catch this.
- **Some readings are judgement calls.** For example, a description with both a volume and a length is flagged and reports the volume.
- **The review rate and the timings are assumptions**, not measurements.